# ML-04 — Refresh / Content Opportunity Data Contract

This notebook defines my Lane 2 slice, checks it against the warehouse, builds five pre-decision features, and deliberately demonstrates one leakage trap. The final June 2026 sample is not used.

## 1. Unit of analysis + time window

### The contract — five answers in plain words

1. **What one row means:** one pseudonymized content page at the **2026-04-01 decision moment**, eligible for a refresh-review ranking.
2. **Tables:** `fact_content_daily_performance` supplies daily search signals. I read only its `month=2026-03` partition for features and its `month=2026-04` partition for the later proxy outcome.
3. **Time window:** features use **2026-03-01 through 2026-03-31**; the proxy label uses **2026-04-01 through 2026-04-30**. A page must have at least 20 fact days in each month and at least 100 March impressions.
4. **What I rank:** pages by the estimated chance of a **future visibility decline**, proxied by April average daily impressions being at least 20% below March average daily impressions. This is a review priority signal, not proof that a refresh will help.
5. **What I deliberately exclude:** every April measurement—especially the April-versus-March impression change—from the honest feature list, because it is only known after the decision and directly defines the proxy label.

**Output:** a directional, evidence-backed queue of visible pages for a human content strategist to review first.

In [1]:
# Safe setup: use a Colab Secret or environment variable named HF_TOKEN.
# The token is never printed or stored in this notebook.
import getpass
import importlib.util
import os
import re
import subprocess
import sys
import warnings

required = {
    "duckdb": "duckdb",
    "huggingface_hub": "huggingface_hub",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

warnings.filterwarnings("ignore", message="IProgress not found.*")
import duckdb
import numpy as np
import pandas as pd
from huggingface_hub import get_token
from IPython.display import display

hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except (ImportError, KeyError, TypeError):
        pass
if not hf_token:
    hf_token = get_token()  # supports a prior local `hf auth login`; the value is not displayed
if not hf_token:
    hf_token = getpass.getpass("Hugging Face READ token (input hidden): ")
if not (hf_token and hf_token.startswith("hf_") and len(hf_token) > 20 and not re.search(r"\s", hf_token)):
    raise RuntimeError("A valid Hugging Face READ token is required; do not paste it into a code cell.")

sql_safe_token = hf_token.replace("'", "''")
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{sql_safe_token}')")
del hf_token
del sql_safe_token

REL = "hf://datasets/FlyRank/internship-warehouse"
MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
APRIL = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"
DECISION_DATE = pd.Timestamp("2026-04-01")

print("Connected securely. Feature month: 2026-03 | outcome month: 2026-04 | decision: 2026-04-01")

Connected securely. Feature month: 2026-03 | outcome month: 2026-04 | decision: 2026-04-01


## 2. Fields: feature / label / context / excluded

| Bucket | Fields used | Rule |
|---|---|---|
| **Features (exactly five)** | `log_march_impressions`, `march_ctr`, `march_weighted_position`, `march_impression_day_share`, `march_position_volatility` | Computed only from March rows, so they exist before 2026-04-01. |
| **Label / proxy** | `future_visibility_decline` | 1 when April average daily impressions are at least 20% below March; 0 otherwise. It is an observed future proxy, not a refresh-effect label. |
| **Context** | `client_hash_id`, `content_hash_id`, decision date, fact-day counts | Used for joins, grouped splitting, coverage checks, and interpretation; never model inputs. |
| **Excluded** | all April measures, `future_impression_change_pct`, hash IDs as model inputs, GA4 metrics, raw/private fields, product decision flags | Future/label-derived values leak; IDs do not carry portable meaning; GA4 coverage is incomplete in this slice; private fields and product decisions are not valid discovery features. |

**Missing values:** pages with fewer than 20 fact days in either month or fewer than 100 March impressions are outside this contract. A missing March position summary is median-imputed inside the training fold; `march_position_volatility` is set to zero only when a page has too few measurable position days for variation to be defined. The availability check below shows why GA4 fields are not among the first five features.

## 3. Verify it with exactly three small queries

These are the three verification queries for the March 2026 slice. Their displayed outputs are aggregate and public-safe.

In [2]:
# Verification query 1 of 3 — grain.
# The source must be unique at page-day grain; aggregating it must be unique at page-decision grain.
grain_sql = f"""
WITH source_duplicates AS (
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS rows_at_grain
    FROM {MARCH}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
), page_decisions AS (
    SELECT client_hash_id, content_hash_id, DATE '2026-04-01' AS decision_date
    FROM {MARCH}
    GROUP BY 1, 2, 3
), decision_duplicates AS (
    SELECT client_hash_id, content_hash_id, decision_date, COUNT(*) AS rows_at_grain
    FROM page_decisions
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
)
SELECT
    (SELECT COUNT(*) FROM source_duplicates) AS duplicate_page_days,
    (SELECT COUNT(*) FROM decision_duplicates) AS duplicate_page_decisions
"""
grain_check = con.sql(grain_sql).df()
display(grain_check)
assert grain_check.iloc[0].eq(0).all(), "The declared grain does not hold."
print("PASS: the March source is one row per page-day and the lane slice is one row per page-decision.")

,duplicate_page_days,duplicate_page_decisions
0,0,0


PASS: the March source is one row per page-day and the lane slice is one row per page-decision.


In [3]:
# Verification query 2 of 3 — row count and date span.
count_window_sql = f"""
SELECT
    COUNT(*) AS daily_rows,
    COUNT(DISTINCT struct_pack(
        client_hash_id := client_hash_id,
        content_hash_id := content_hash_id
    )) AS distinct_pages,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM {MARCH}
"""
count_window_check = con.sql(count_window_sql).df()
display(count_window_check)
assert str(count_window_check.loc[0, "first_date"])[:10] == "2026-03-01"
assert str(count_window_check.loc[0, "last_date"])[:10] == "2026-03-31"
print("PASS: the requested middle-panel month spans all of March 2026.")

,daily_rows,distinct_pages,first_date,last_date
0,9841378,331437,2026-03-01,2026-03-31


PASS: the requested middle-panel month spans all of March 2026.


In [4]:
# Verification query 3 of 3 — availability.
# IS TRUE is intentional: pre-GA4 zero-filled rows are not treated as measured zero engagement.
availability_sql = f"""
WITH all_rows AS (
    SELECT client_hash_id, content_hash_id
    FROM {MARCH}
), available_rows AS (
    SELECT client_hash_id, content_hash_id
    FROM {MARCH}
    WHERE ga4_data_available IS TRUE
)
SELECT
    (SELECT COUNT(*) FROM all_rows) AS all_daily_rows,
    (SELECT COUNT(*) FROM available_rows) AS ga4_available_daily_rows,
    ROUND(
        (SELECT COUNT(*) FROM available_rows) * 100.0 /
        NULLIF((SELECT COUNT(*) FROM all_rows), 0),
        2
    ) AS rows_surviving_pct
"""
availability_check = con.sql(availability_sql).df()
display(availability_check)
assert availability_check.loc[0, "ga4_available_daily_rows"] <= availability_check.loc[0, "all_daily_rows"]
print("PASS: GA4 availability was filtered with IS TRUE; only measured rows survive.")

verification_queries = [grain_sql, count_window_sql, availability_sql]
assert len(verification_queries) == 3

,all_daily_rows,ga4_available_daily_rows,rows_surviving_pct
0,9841378,413966,4.21


PASS: GA4 availability was filtered with IS TRUE; only measured rows survive.


### Five-feature frame

The next SQL statement builds the modeling frame; it is separate from the three verification queries above. It reads March for features and April only for the proxy outcome.

In [5]:
feature_frame_sql = f"""
WITH march_page AS (
    SELECT
        client_hash_id,
        content_hash_id,
        COUNT(*) AS march_fact_days,
        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_clicks) AS march_clicks,
        COUNT(*) FILTER (WHERE gsc_impressions > 0) AS march_impression_days,
        SUM(CASE
            WHEN gsc_impressions > 0 AND gsc_avg_position > 0
            THEN gsc_avg_position * gsc_impressions ELSE 0
        END) AS march_position_weighted_sum,
        SUM(CASE
            WHEN gsc_impressions > 0 AND gsc_avg_position > 0
            THEN gsc_impressions ELSE 0
        END) AS march_position_weight,
        STDDEV_SAMP(gsc_avg_position) FILTER (
            WHERE gsc_impressions > 0 AND gsc_avg_position > 0
        ) AS march_position_sd
    FROM {MARCH}
    GROUP BY 1, 2
), april_page AS (
    SELECT
        client_hash_id,
        content_hash_id,
        COUNT(*) AS april_fact_days,
        SUM(gsc_impressions) AS april_impressions
    FROM {APRIL}
    GROUP BY 1, 2
), eligible AS (
    SELECT
        m.*,
        a.april_fact_days,
        a.april_impressions,
        (a.april_impressions::DOUBLE / a.april_fact_days) /
        NULLIF(m.march_impressions::DOUBLE / m.march_fact_days, 0) - 1.0
            AS future_impression_change_pct
    FROM march_page AS m
    INNER JOIN april_page AS a USING (client_hash_id, content_hash_id)
    WHERE m.march_fact_days >= 20
      AND a.april_fact_days >= 20
      AND m.march_impressions >= 100
)
SELECT
    client_hash_id,
    content_hash_id,
    LN(1.0 + march_impressions) AS log_march_impressions,
    march_clicks::DOUBLE / NULLIF(march_impressions, 0) AS march_ctr,
    march_position_weighted_sum / NULLIF(march_position_weight, 0)
        AS march_weighted_position,
    march_impression_days::DOUBLE / march_fact_days AS march_impression_day_share,
    COALESCE(march_position_sd, 0.0) AS march_position_volatility,
    CASE WHEN future_impression_change_pct <= -0.20 THEN 1 ELSE 0 END
        AS future_visibility_decline,
    future_impression_change_pct AS _label_source_for_leak_demo
FROM eligible
"""

model_source = con.sql(feature_frame_sql).df()
feature_cols = [
    "log_march_impressions",
    "march_ctr",
    "march_weighted_position",
    "march_impression_day_share",
    "march_position_volatility",
]
context_cols = ["client_hash_id", "content_hash_id"]
label_col = "future_visibility_decline"

feature_frame = model_source[context_cols + feature_cols + [label_col]].copy()
future_change_for_demo = model_source["_label_source_for_leak_demo"].copy()

assert len(feature_cols) == 5
assert not feature_frame.duplicated(context_cols).any()
assert "_label_source_for_leak_demo" not in feature_frame.columns
print(f"Feature frame: {len(feature_frame):,} eligible page-decisions x {len(feature_cols)} honest features")
print(f"Observed future-decline proxy rate: {feature_frame[label_col].mean():.1%}")
display(feature_frame[feature_cols + [label_col]].head())

Feature frame: 100,049 eligible page-decisions x 5 honest features
Observed future-decline proxy rate: 50.5%


,log_march_impressions,march_ctr,march_weighted_position,march_impression_day_share,march_position_volatility,future_visibility_decline
0,6.774224,0.002288,6.425629,1.000000,3.212244,1
1,7.028201,0.000000,5.023957,1.000000,1.199715,1
2,6.115892,0.002212,7.542035,0.967742,3.454185,1
3,7.386471,0.000620,5.921885,1.000000,2.711968,1
4,6.648985,0.002594,6.814527,1.000000,1.486444,1


### Available when? — one line per feature

- **`log_march_impressions`** — Knowable at the decision moment because all March impression rows have landed before 2026-04-01; the log only rescales that known total.
- **`march_ctr`** — Knowable at the decision moment because it uses only March clicks divided by March impressions.
- **`march_weighted_position`** — Knowable at the decision moment because it aggregates only March GSC position observations, weighted by their March impressions.
- **`march_impression_day_share`** — Knowable at the decision moment because it counts March days with observed impressions and divides by March fact days.
- **`march_position_volatility`** — Knowable at the decision moment because it measures variation only among daily March position observations.

### The trap — add one label-derived column, score it, then delete it

I first score the five honest features on clients held out from training. Then I add exactly one forbidden column: the April-versus-March impression change that directly defines the label. A very high score from that column is evidence of leakage, not a discovery.

In [6]:
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier

groups = feature_frame["client_hash_id"]
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(feature_frame, feature_frame[label_col], groups))
y_train = feature_frame.iloc[train_idx][label_col]
y_test = feature_frame.iloc[test_idx][label_col]
assert y_train.nunique() == 2 and y_test.nunique() == 2, "Both splits need both label classes."

def quick_tree():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42),
    )

honest_model = quick_tree()
honest_model.fit(feature_frame.iloc[train_idx][feature_cols], y_train)
honest_auc = roc_auc_score(
    y_test,
    honest_model.predict_proba(feature_frame.iloc[test_idx][feature_cols])[:, 1],
)

# Deliberately spring the trap: add ONE label-derived future column.
leaky_col = "future_impression_change_pct"
leaky_frame = feature_frame.copy()
leaky_frame[leaky_col] = future_change_for_demo.to_numpy()
leaky_features = feature_cols + [leaky_col]
assert len(set(leaky_features) - set(feature_cols)) == 1

leaky_model = quick_tree()
leaky_model.fit(leaky_frame.iloc[train_idx][leaky_features], y_train)
leaky_auc = roc_auc_score(
    y_test,
    leaky_model.predict_proba(leaky_frame.iloc[test_idx][leaky_features])[:, 1],
)

score_table = pd.DataFrame({
    "held-out client score": ["Honest five-feature ROC AUC", "Leaky ROC AUC"],
    "value": [honest_auc, leaky_auc],
})
score_table["value"] = score_table["value"].round(3)
display(score_table)
print(f"The apparent gain is {leaky_auc - honest_auc:+.3f}; it comes from seeing the outcome, so it is invalid.")

# Delete the trap and retain the honest result.
del leaky_frame[leaky_col]
del leaky_features
assert leaky_col not in leaky_frame.columns
assert feature_cols == [
    "log_march_impressions",
    "march_ctr",
    "march_weighted_position",
    "march_impression_day_share",
    "march_position_volatility",
]
print(f"Leak deleted. Honest held-out-client ROC AUC retained: {honest_auc:.3f}")

,held-out client score,value
0,Honest five-feature ROC AUC,0.648
1,Leaky ROC AUC,1.000


The apparent gain is +0.352; it comes from seeing the outcome, so it is invalid.
Leak deleted. Honest held-out-client ROC AUC retained: 0.648


## 4. Data limits

**Named limitation — visible-page coverage bias:** this slice deliberately keeps pages with at least 100 March impressions and at least 20 fact days in both March and April. Its numbers therefore do not describe new, low-volume, or short-history pages well. The unbalanced client panel can also mix true page movement with client-level seasonality or demand shifts.

More fundamentally, the warehouse records what happened; it does not record a randomized refresh treatment. A high rank can support a human review decision, but it cannot show that refreshing a page would cause recovery.

In [7]:
# Compact machine-check of the assignment contract.
assert len(verification_queries) == 3
assert "IS TRUE" in availability_sql.upper()
assert len(feature_cols) == 5
assert leaky_col not in feature_frame.columns
assert not feature_frame.duplicated(context_cols).any()
assert feature_frame[label_col].isin([0, 1]).all()
print("SELF-CHECK PASS: 3 verification queries | 5 honest features | leak removed | one row per page-decision")

SELF-CHECK PASS: 3 verification queries | 5 honest features | leak removed | one row per page-decision


## 5. Self-check

- [x] Five plain-words contract answers are present.
- [x] Exactly three verification queries are executed with visible outputs.
- [x] Availability is checked with `ga4_data_available IS TRUE`.
- [x] The frame has exactly five features and an “available when?” line for each.
- [x] One label-derived leak is added, scored, deleted, and the honest score is kept.
- [x] One named limitation is stated.
- [x] No client names, URLs, private queries, raw data, or token values are displayed.
- [x] Claims use careful words: observed, proxy, directional, decision-support.